# DTL: Feature Extraction and Generate Probability Layer using OVR method

# 1. Import satellite image from asset

In [1]:
!python -m pip install .. --quiet

VERSION = 'v7'
REGION = 'Sumatra'
region_lower = REGION.lower()

import ee 

ee.Authenticate() 
ee.Initialize(project='epistem2')


In [2]:

# aoi definition

# import region from Hadi's assets
# region_name = "Sumatera"
# regions_fc = ee.FeatureCollection("users/hadicu06/IIASA/RESTORE/vector_datasets/classification_regions")
# aoi = regions_fc.filter(ee.Filter.eq('region_name', region_name)).geometry()

# try province export (aceh)

provinces = ee.FeatureCollection(f'projects/epistem2/assets/AOI_{REGION}_Provinces')
province_list = provinces.toList(provinces.size())
aoi = ee.Feature(province_list.get(0)).geometry()

# Load satellite image stack from asset

final_stack = ee.Image(f'projects/epistem2/assets/stacked_landsat_pred_2021_{region_lower}_{VERSION}')

# 2. Classification scheme 

In [3]:
from luma_ge.classification_scheme import LULC_Scheme_Manager

manager = LULC_Scheme_Manager()

scheme_name = "Epistem"
success, message = manager.load_default_scheme(scheme_name)
classification_df = manager.get_dataframe()

print(classification_df.to_string(index=False))

 ID          Land Cover Class Color Palette
  1    Primary Dryland Forest       #006400
  2  Secondary Dryland Forest       #228B22
  3   Primary Mangrove Forest       #4169E1
  4 Secondary Mangrove Forest       #87CEEB
  5      Primary Swamp Forest       #2E8B57
  6    Secondary Swamp Forest       #8FBC8F
  7         Plantation Forest       #32CD32
  8        Rubber Monoculture       #8B4513
  9      Oil palm Monoculture       #FF8C00
 10         Cacao Monoculture       #D2691E
 11       Coconut monoculture       #F4A460
 12         Other Monoculture       #DAA520
 13            Other Cropland       #FFFF00
 14       Coffee agroforestry       #6B8E23
 15       Rubber agroforestry       #9ACD32
 16         Mixed/home garden       #7CFC00
 17               Paddy field       #EEE8AA
 18          Grass or Savanna       #ADFF2F
 19                     Shrub       #90EE90
 20                Settlement       #FF0000
 21              Cleared Land       #D2B48C
 22               Mining area   

# 3. Load labelled training data from local

In [6]:
import geopandas as gpd
from shapely.geometry import shape
import geemap

TrainVectPath = f"../data/modular_mapping_approach/{region_lower}_test/{region_lower}_td_DTL_result_{VERSION}.shp"
TrainData = gpd.read_file(TrainVectPath)
if TrainData.crs is None:
    TrainData = TrainData.set_crs("EPSG:4326")
aoi_geojson = aoi.getInfo()
aoi_geom = shape(aoi_geojson)
aoi_gdf = gpd.GeoDataFrame(
    {"geometry": [aoi_geom]},
    crs="EPSG:4326"
)
aoi_gdf = aoi_gdf.to_crs(TrainData.crs)


# REMOVE UNCLASSIFIED CLASS
TrainDataFinal = TrainData[TrainData["label"] != 0].copy()
    
TrainDataFinal = gpd.clip(
    TrainDataFinal,
    aoi_gdf
)

print(f"Total training points : {len(TrainData)}")
print(f"Points inside AOI     : {len(TrainDataFinal)}")
print(f"Points removed        : {len(TrainData) - len(TrainDataFinal)}")
print(TrainData.columns.tolist())

Total training points : 9798
Points inside AOI     : 711
Points removed        : 9087
['AoI', 'ID', 'LULC_24', 'agricultur', 'artificial', 'bareSoil_c', 'builtup_co', 'cacao_pres', 'coconut_pr', 'coffee_pre', 'mangrove_p', 'mining', 'oilpalm_pr', 'paddy_pres', 'rubber_pre', 'timber_ext', 'tree_cover', 'tree_heigh', 'waterbody_', 'longitude', 'latitude', 'label', 'class_name', 'geometry']


In [7]:
CLASS_PROPERTY = 'label'

band_names = final_stack.bandNames()
labeled_roi = geemap.gdf_to_ee(TrainDataFinal)

# remove unecessary properties (i.e. primitives) from the training and testing datasets

input_props = band_names.add(CLASS_PROPERTY)
labeled_roi = labeled_roi.select(input_props)

# 4. Feature extraction from satellite imageries

In [ ]:
# import geemap

# Map = geemap.Map()

# Map.centerObject(aoi, 7)
# Map.addLayer(aoi, {'color': 'red'}, 'AOI')
# Map.addLayer(stacked_landsat, {}, 'stacked_landsat_2020_Sumatera')
# Map.addLayer(labeled_roi, {}, 'labeled_roi')

# Map

In [8]:
from luma_ge.classification import FeatureExtraction

feature_extractor = FeatureExtraction()

stratified_train, stratified_test = feature_extractor.stratified_split(
                                                    labeled_roi, 
                                                    final_stack, 
                                                    class_prop=CLASS_PROPERTY, 
                                                    train_ratio=0.5
                                                )

Stratified Random Split Training Pixel Size: 626
Stratified Random Split Testing Pixel Size: 85


## Convert training data from EE object to pandas

In [9]:
# CHANGE FROM EE LIST TO PANDAS LIST

import pandas as pd

train_info = stratified_train.getInfo()

train_features = train_info["features"]

train_data = [
    feature["properties"]
    for feature in train_features
]

train_df = pd.DataFrame(train_data)

feature_names = final_stack.bandNames().getInfo()

train_df = train_df.dropna(
    subset=feature_names + [CLASS_PROPERTY]
)

print("Training samples:", len(train_df))
print("Features:", feature_names)

Training samples: 624
Features: ['AEROSOL', 'BLUE', 'GREEN', 'RED', 'NIR', 'SWIR1', 'SWIR2', 'NDVI', 'MNDWI', 'NDBI']


# 5. Balance the training dataset and train random forest to generate probability layer for each class

Use `RandomUnderSampler` to balance the negative training data in the binary classification

In [10]:
import ee
import pandas as pd
from imblearn.under_sampling import RandomUnderSampler

prob_images = []
class_list = sorted(train_df[CLASS_PROPERTY].unique().tolist())

for class_id in class_list:

    # Create a one-vs-rest target for the current class.
    binary_target = (train_df[CLASS_PROPERTY] == class_id).astype(int)

    # Balance the positive and negative samples before sending them to EE.
    rus = RandomUnderSampler(random_state=42)
    X_resampled, y_resampled = rus.fit_resample(
        train_df[feature_names],
        binary_target
    )
    print(f"Class {class_id}: training points after undersampling = {len(X_resampled)}")
    print(f"Class non-{class_id}: training points after undersampling = {len(y_resampled)}")

    balanced_df = X_resampled.copy()
    balanced_df["binary"] = y_resampled.to_numpy()
    balanced_records = balanced_df.to_dict(orient="records")

    # Convert numpy scalar values to native Python values for EE serialization.
    balanced_features = [
        ee.Feature(
            None,
            {
                name: value.item() if hasattr(value, "item") else value
                for name, value in record.items()
            }
        )
        for record in balanced_records
    ]
    balanced_training = ee.FeatureCollection(balanced_features)

    # Train a native Earth Engine classifier in probability output mode.
    ee_classifier = ee.Classifier.smileRandomForest(
        numberOfTrees=100,
        variablesPerSplit=1,
        minLeafPopulation=1,
        seed=42
    ).setOutputMode("PROBABILITY").train(
        features=balanced_training,
        classProperty="binary",
        inputProperties=feature_names
    )

    probability_image = (
        final_stack
        .select(feature_names)
        .classify(ee_classifier)
        .multiply(100)  # Scale probabilities to 0-100 range
        .rename(f"prob_{class_id}")
    )
    prob_images.append(probability_image)

# Stack one probability band for every class.
prob_stack = prob_images[0]
for probability_image in prob_images[1:]:
    prob_stack = prob_stack.addBands(probability_image)

print("Probability bands:")
print(prob_stack.bandNames().getInfo())

Class 1: training points after undersampling = 256
Class non-1: training points after undersampling = 256
Class 2: training points after undersampling = 452
Class non-2: training points after undersampling = 452
Class 3: training points after undersampling = 18
Class non-3: training points after undersampling = 18
Class 4: training points after undersampling = 136
Class non-4: training points after undersampling = 136
Class 5: training points after undersampling = 18
Class non-5: training points after undersampling = 18
Class 6: training points after undersampling = 54
Class non-6: training points after undersampling = 54
Class 7: training points after undersampling = 2
Class non-7: training points after undersampling = 2
Class 8: training points after undersampling = 2
Class non-8: training points after undersampling = 2
Class 9: training points after undersampling = 42
Class non-9: training points after undersampling = 42
Class 13: training points after undersampling = 6
Class non-13

In [ ]:
# sanity check

#1. Check number and names of probability bands
print("Bands:", prob_stack.bandNames().getInfo())
print("Number of bands:", prob_stack.bandNames().size().getInfo())

# 2. Check pixel type (should be Byte because of .byte())
print("Image type:", prob_stack.bandTypes().getInfo())

# 3. Check a single pixel only
sample = prob_stack.sample(
    region=aoi.centroid(),
    scale=aoi.projection().nominalScale(),
    numPixels=1,
    geometries=False
)

print("Single-pixel probability values:")
print(sample.first().getInfo())

In [ ]:
# Export the probability stack to an asset

# Export to Earth Engine Asset
# task = ee.batch.Export.image.toAsset(
#     image=prob_stack,
#     description=f'probability_stack_{REGION}_2021_{VERSION}',
#     assetId=f'projects/epistem2/assets/probability_stack_{REGION}_2021_{VERSION}',
#     region=aoi,
#     scale=100,
#     maxPixels=1e13
# )

# task.start()